# 🧹 Pipeline de Limpieza Estadística, Integración y Puerta de Calidad (Capas Bronce $\rightarrow$ Plata $\rightarrow$ Oro)

**Proyecto Integrador — Entrega E1 (Unidad 3)**  
**Observatorio Analítico del Mercado Inmobiliario y Renta Habitacional en la ZMVM**  
**Integrantes del Equipo:**
- Juan Antonio Ortiz Hinojosa
- Diego Alberto Virgen Cuéllar

---

## 🎯 Alcance del Notebook (`docs/E1/limpieza.ipynb`)
Este cuaderno ejecutable documenta y verifica paso a paso el flujo automatizado en `src/pipeline_e1.py`:
1. **Integración y Resolución de Conflictos:** Fusión relacional entre indicadores macroeconómicos de BANXICO/INEGI y ofertas inmobiliarias de la ZMVM, homologando fechas (`DD/MM/YYYY` $\rightarrow$ `YYYY-MM`), formatos monetarios en texto (`"$25,400.00 MXN"` $\rightarrow$ `float`) y etiquetas categóricas.
2. **Puerta de Calidad Automática (`Pandera`) y Cuarentena:** Validación formal de esquemas, tipos, unicidad y rangos físicos. Los registros inválidos se envían con su motivo exacto a `data/silver/inmobiliario/cuarentena_inmobiliario.csv` (**nunca se borran en silencio**).
3. **Valores Faltantes (Diagnóstico MCAR / MAR / MNAR e Imputación Justificada):** Pruebas de hipótesis ($\chi^2$ y $t$ de Welch) e imputación diferenciada según el mecanismo generador.
4. **Detección y Tratamiento de Atípicos y Ruido:** Comparación de 3 métodos (**IQR**, **Z-Score Robusto con MAD** e **Isolation Forest** multivariado) y tratamiento vía **Winsorización estratificada** ($P_{1\%}-P_{99\%}$).
5. **Normalización y Estandarización:** Transformaciones **Min-Max** $[0,1]$ y **Z-Score** ($\mu=0, \sigma=1$).
6. **Discretización y Jerarquías Conceptuales:** Categorización ordinal de metrajes, cuartiles de precio por $m^2$, velocidad de absorción y jerarquías espacial/temporal/tipológica.
7. **Reducción de Datos hacia Capa Oro:** Selección de variables (reducción de dimensionalidad del $49.18\%$) y construcción de los 3 cubos analíticos en `data/gold/inmobiliario/` para responder `E1`, `E2` y `E3`.

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np

# Agregar directorio raíz del proyecto al path para importar src.pipeline_e1
PROJECT_ROOT = Path.cwd().resolve()
if (PROJECT_ROOT / "src").exists():
    ROOT_DIR = PROJECT_ROOT
else:
    ROOT_DIR = PROJECT_ROOT.parent.parent
sys.path.insert(0, str(ROOT_DIR))

from src.pipeline_e1 import (
    load_and_integrate_sources,
    run_quality_gate_with_quarantine,
    diagnose_and_impute_missing_values,
    detect_and_treat_outliers,
    transform_scale_discretize_and_hierarchies,
    build_gold_layer_datasets,
)

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 120)
print("Raíz del proyecto cargada correctamente:", ROOT_DIR)

Raíz del proyecto cargada correctamente: C:\Users\Colibecas\.gemini\antigravity\scratch\observatorio-inmobiliario


## 1. Integración de Fuentes Heterogéneas y Resolución de Conflictos
Se integran las dos fuentes primarias capturadas en la Capa Bronce (`data/bronze/inmobiliario/`):
- **Fuente 1 (`macro_banxico_inegi_36m.json`):** 36 meses consecutivos de INPC General, INPC Vivienda ZMVM, TIIE a 28 días y Tipo de Cambio FIX.
- **Fuente 2 (`rentas_zmvm_historico_36m.json`):** 1,299 registros crudos de ofertas de renta residencial en la ZMVM.

**Conflictos resueltos antes de la unión:**
1. Fechas crudas en formato BANXICO (`DD/MM/YYYY`) normalizadas al estándar ISO mensual (`YYYY-MM`).
2. Precios con símbolos monetarios y comas (`"$32,500.00 MXN"`) parseados a flotantes IEEE 754.
3. Categorías de inmueble heterogéneas (`"Depto 1 Rec"`, `"Departamento 2 Recámaras"`, `"CASA HABITACION"`) mapeadas al catálogo canónico (`depto_1rec`, `depto_2rec`, `casa`).

In [2]:
df_integrated, integration_report = load_and_integrate_sources()
print("=== Reporte de Integración y Resolución de Conflictos ===")
print(json.dumps(integration_report, indent=2, ensure_ascii=False))
df_integrated[["id_propiedad", "periodo_mensual", "subzona_id", "tipo_inmueble_raw", "tipo_inmueble", "superficie_m2", "renta_lista_mensual_mxn", "inpc_vivienda_zmvm", "tiie_28d_pct"]].head(8)

=== Reporte de Integración y Resolución de Conflictos ===
{
  "total_registros_bronce": 1299,
  "total_periodos_macro": 36,
  "registros_integrados": 1299,
  "conflictos_moneda_string_resueltos": 37,
  "conflictos_categoria_tipo_inmueble_resueltos": 48,
  "fechas_banxico_normalizadas": 36,
  "registros_sin_match_macro": 0
}


,id_propiedad,periodo_mensual,subzona_id,tipo_inmueble_raw,tipo_inmueble,superficie_m2,renta_lista_mensual_mxn,inpc_vivienda_zmvm,tiie_28d_pct
0,PROP-ACC81F8D42,2023-11,SZ1_PRIME_PONIENTE,Depto 1 Rec,depto_1rec,63.7,27810.0,121.4,11.5
1,PROP-4E2A0FC1CC,2023-11,SZ1_PRIME_PONIENTE,depto_1rec,depto_1rec,63.7,29730.0,121.4,11.5
2,PROP-898B72BF4C,2023-11,SZ1_PRIME_PONIENTE,depto_1rec,depto_1rec,50.9,22480.0,121.4,11.5
3,PROP-A7FA0E3263,2023-11,SZ1_PRIME_PONIENTE,Departamento 2 Recámaras,depto_2rec,90.6,36800.0,121.4,11.5
4,PROP-39E1C13A7F,2023-11,SZ1_PRIME_PONIENTE,depto_2rec,depto_2rec,107.8,44050.0,121.4,11.5
5,PROP-499C016032,2023-11,SZ1_PRIME_PONIENTE,depto_2rec,depto_2rec,71.1,46070.0,121.4,11.5
6,PROP-D17B7B19BA,2023-11,SZ1_PRIME_PONIENTE,CASA HABITACION,casa,186.2,63340.0,121.4,11.5
7,PROP-7CE310881F,2023-11,SZ1_PRIME_PONIENTE,casa,casa,241.9,83460.0,121.4,11.5


## 2. Puerta de Calidad Automática (`Pandera`) con Cuarentena Documentada
Antes de promover los registros a la Capa Plata, ejecutamos el contrato declarativo con `pandera`:
- **Unicidad compuesta:** `(id_propiedad, periodo_mensual)` sin duplicados por republicación de corredores.
- **Rangos físicos plausibles:** `superficie_m2` $\in [20.0, 1200.0]\text{ m}^2$ (permitiendo nulos transitorios para imputación MAR).
- **Precios estrictamente válidos:** `renta_lista_mensual_mxn` $\in (1500.0, 500000.0]\text{ MXN}$ y no nulo.
- **Política de Cuarentena:** Los registros que violan cualquier regla **nunca se eliminan en silencio**; se envían a `data/silver/inmobiliario/cuarentena_inmobiliario.csv` con la columna `motivo_cuarentena`.

In [3]:
df_valid, df_quarantine, gate_report = run_quality_gate_with_quarantine(df_integrated)
print("=== Resultado de la Puerta de Calidad (Pandera) ===")
print(json.dumps(gate_report, indent=2, ensure_ascii=False))
print("\n=== Registros Enviados a Cuarentena (100% Trazables) ===")
df_quarantine[["id_propiedad", "periodo_mensual", "subzona_id", "tipo_inmueble", "superficie_m2", "renta_lista_mensual_mxn", "motivo_cuarentena"]]

=== Resultado de la Puerta de Calidad (Pandera) ===
{
  "total_evaluados": 1299,
  "total_aprobados_plata": 1291,
  "total_enviados_cuarentena": 8,
  "tasa_rechazo_cuarentena_pct": 0.62,
  "desglose_motivos_cuarentena": {
    "DUPLICADO_ID_PERIODO: Registro duplicado por republicación en el mismo mes": 3,
    "PANDERA_FAIL[superficie_m2]: superficie_m2_menor_20m2 (valor=-25.0)": 1,
    "PANDERA_FAIL[superficie_m2]: superficie_m2_mayor_1200m2 (valor=9850.0)": 1,
    "PANDERA_FAIL[renta_lista_mensual_mxn]: renta_lista_invalida_o_negativa (valor=-15000.0)": 1,
    "PANDERA_FAIL[renta_lista_mensual_mxn]: not_nullable (valor=nan)": 1,
    "PANDERA_FAIL[superficie_m2]: superficie_m2_menor_20m2 (valor=4.5)": 1
  }
}

=== Registros Enviados a Cuarentena (100% Trazables) ===


,id_propiedad,periodo_mensual,subzona_id,tipo_inmueble,superficie_m2,renta_lista_mensual_mxn,motivo_cuarentena
14,PROP-55948A2DE0,2023-11,SZ2_CENTRAL_CORREDOR,depto_2rec,92.5,34390.0,DUPLICADO_ID_PERIODO: Registro duplicado por r...
190,PROP-C3DE9C8393,2024-04,SZ2_CENTRAL_CORREDOR,depto_1rec,-25.0,19920.0,PANDERA_FAIL[superficie_m2]: superficie_m2_men...
447,PROP-63EC2E6AF5,2024-11,SZ2_CENTRAL_CORREDOR,depto_2rec,71.0,27910.0,DUPLICADO_ID_PERIODO: Registro duplicado por r...
459,PROP-32AEE09250,2024-11,SZ3_CONSOLIDADA_SUR,casa,9850.0,49850.0,PANDERA_FAIL[superficie_m2]: superficie_m2_may...
689,PROP-8BC19A09A2,2025-06,SZ1_PRIME_PONIENTE,depto_2rec,89.8,-15000.0,PANDERA_FAIL[renta_lista_mensual_mxn]: renta_l...
880,PROP-D3D02319DE,2025-11,SZ2_CENTRAL_CORREDOR,depto_2rec,98.3,41340.0,DUPLICADO_ID_PERIODO: Registro duplicado por r...
931,PROP-AF3AB2A700,2025-12,SZ4_EMERGENTE_METROPOLITANA,depto_1rec,54.7,NaN,PANDERA_FAIL[renta_lista_mensual_mxn]: not_nul...
1133,PROP-DD88349678,2026-06,SZ2_CENTRAL_CORREDOR,depto_2rec,4.5,39780.0,PANDERA_FAIL[superficie_m2]: superficie_m2_men...


## 3. Diagnóstico del Mecanismo de Valores Faltantes (MCAR / MAR / MNAR) e Imputación
Sobre los $1,291$ registros que aprobaron la Puerta de Calidad se evalúan estadísticamente las 3 columnas con valores faltantes:
1. **`estacionamientos` ($28$ faltantes — MCAR):** La prueba $\chi^2$ de independencia contra `subzona_id` arroja $p = 1.00 > 0.05$. Al ser completamente aleatorio (**MCAR**), se imputa mediante la **moda condicional por `tipo_inmueble`**.
2. **`superficie_m2` ($15$ faltantes — MAR):** La prueba $\chi^2$ contra `subzona_id` arroja $\chi^2 = 45.48, p = 7.31 \times 10^{-10} < 0.001$. La probabilidad de omisión depende de la sub-zona observada (`SZ4_EMERGENTE_METROPOLITANA`). Al ser **MAR**, se imputa con la **mediana estratificada por `(subzona_id, tipo_inmueble, recamaras)`**.
3. **`cuota_mantenimiento_mxn` ($108$ faltantes — MNAR):** La prueba $t$ de Welch muestra que la renta media cuando falta la cuota es de $\$60,821.11\text{ MXN}$ frente a $\$35,074.77\text{ MXN}$ cuando está presente ($t = 14.67, p = 1.56 \times 10^{-29}$). Los anunciantes omiten estratégicamente cuotas caras (**MNAR**). Se crea el indicador binario `flag_mnar_mantenimiento` y se imputa mediante la **razón mediana zonal (`cuota / renta_lista`)**.

In [4]:
df_imputed, missing_report = diagnose_and_impute_missing_values(df_valid)
print("=== Diagnóstico Estadístico MCAR / MAR / MNAR e Imputación ===")
print(json.dumps(missing_report, indent=2, ensure_ascii=False))

=== Diagnóstico Estadístico MCAR / MAR / MNAR e Imputación ===
{
  "faltantes_antes_imputacion": {
    "estacionamientos": 28,
    "superficie_m2": 15,
    "cuota_mantenimiento_mxn": 108
  },
  "faltantes_despues_imputacion": {
    "estacionamientos": 0,
    "superficie_m2": 0,
    "cuota_mantenimiento_mxn": 0
  },
  "diagnostico_mecanismos": {
    "estacionamientos_MCAR": {
      "mecanismo": "MCAR (Missing Completely At Random)",
      "chi2_stat": 0.0001,
      "p_value": 1.0,
      "interpretacion": "p > 0.05 confirma que la ausencia no depende de la ubicación ni del precio; se imputa por moda condicional del tipo de inmueble."
    },
    "superficie_m2_MAR": {
      "mecanismo": "MAR (Missing At Random)",
      "chi2_stat": 45.482,
      "p_value": 7.30833849863119e-10,
      "interpretacion": "p < 0.001 demuestra dependencia sistemática respecto a la variable observada subzona_id (concentrado en SZ4); se imputa con mediana estratificada por (subzona_id, tipo_inmueble, recamaras).

## 4. Detección y Tratamiento de Atípicos y Ruido (IQR, Z-Score Robusto e Isolation Forest)
Comparamos tres enfoques complementarios sobre los registros válidos imputados:
- **IQR estratificado por sub-zona:** Detecta valores fuera de $[Q_1 - 1.5\cdot IQR, Q_3 + 1.5\cdot IQR]$.
- **Z-Score Robusto (Mediana y MAD):** $Z_{\text{rob}} = \frac{0.6745(x - \text{Mediana})}{MAD}$, marcando $|Z_{\text{rob}}| > 3.5$.
- **Isolation Forest Multivariado (`scikit-learn`):** Evalúa conjuntamente `precio_m2_mxn`, `superficie_m2` y `yield_bruto_anual_pct` con `contamination=0.025`.
- **Tratamiento:** Se aplica **Winsorización estratificada** en los percentiles $[P_{1\%}, P_{99\%}]$ por `subzona_id` (`precio_m2_winsor_mxn`), reduciendo la desviación estándar sin amputar observaciones reales del corredor Prime.

In [5]:
df_outliers_treated, outlier_report = detect_and_treat_outliers(df_imputed)
print("=== Reporte de Detección y Tratamiento de Atípicos ===")
print(json.dumps(outlier_report, indent=2, ensure_ascii=False))

# Mostrar los registros detectados por consenso (>= 2 métodos)
df_outliers_treated[df_outliers_treated["outlier_consenso"] == 1][
    ["id_propiedad", "periodo_mensual", "subzona_id", "tipo_inmueble", "precio_m2_mxn", "precio_m2_winsor_mxn", "zscore_robusto_precio_m2", "outlier_iqr", "outlier_zrobusto", "outlier_isolation_forest"]
]

=== Reporte de Detección y Tratamiento de Atípicos ===
{
  "detectados_iqr": 4,
  "detectados_zscore_robusto": 3,
  "detectados_isolation_forest": 33,
  "detectados_consenso_2_de_3": 4,
  "precio_m2_max_antes_winsor": 765.72,
  "precio_m2_max_despues_winsor": 626.96,
  "precio_m2_std_antes_winsor": 89.7,
  "precio_m2_std_despues_winsor": 88.63
}


,id_propiedad,periodo_mensual,subzona_id,tipo_inmueble,precio_m2_mxn,precio_m2_winsor_mxn,zscore_robusto_precio_m2,outlier_iqr,outlier_zrobusto,outlier_isolation_forest
5,PROP-499C016032,2023-11,SZ1_PRIME_PONIENTE,depto_2rec,647.96,626.96,3.1088,1,0,1
328,PROP-162B5C4896,2024-08,SZ1_PRIME_PONIENTE,depto_2rec,717.28,626.96,4.1272,1,1,1
651,PROP-E3023C1358,2025-05,SZ1_PRIME_PONIENTE,depto_2rec,689.91,626.96,3.7251,1,1,1
973,PROP-B07FA0DFF1,2026-02,SZ1_PRIME_PONIENTE,depto_2rec,765.72,626.96,4.8389,1,1,1


## 5. Normalización, Estandarización, Discretización y Jerarquías Conceptuales
1. **Normalización Min-Max $[0, 1]$ y Estandarización Z-Score ($\mu=0, \sigma=1$):** Aplicadas sobre `precio_m2_winsor_mxn`, `superficie_m2`, `renta_efectiva_winsor_mxn`, `yield_bruto_anual_pct` y `dias_vacancia_observados`.
2. **Discretización Ordinal:**
   - `rango_superficie_cat`: Compacto ($<60\text{ m}^2$), Medio ($60–100\text{ m}^2$), Amplio ($100–160\text{ m}^2$), Residencial Plus ($>160\text{ m}^2$).
   - `estrato_precio_m2_cat`: Cuartiles `Q1_Accesible`, `Q2_Moderado`, `Q3_Alto`, `Q4_Prime`.
   - `nivel_vacancia_cat`: Rápida ($\le 21$ días), Normal ($22–35$ días), Lenta ($36–50$ días), Estancada ($>50$ días).
3. **Jerarquías Conceptuales:**
   - **Espacial:** `Colonia -> Alcaldía/Municipio -> Sub-zona -> Corredor Macro -> Región ZMVM`.
   - **Temporal:** `Periodo Mensual (YYYY-MM) -> Trimestre (YYYY-Q#) -> Año`.
   - **Tipológica:** `Tipo de Inmueble -> Familia (Vivienda Vertical vs. Vivienda Horizontal)`.

In [6]:
df_silver_full, transform_report = transform_scale_discretize_and_hierarchies(df_outliers_treated)
print("=== Resumen de Normalización, Discretización y Jerarquías ===")
print(json.dumps(transform_report, indent=2, ensure_ascii=False))
df_silver_full[[
    "id_propiedad", "jerarquia_ubicacion_completa", "trimestre", "familia_inmueble",
    "precio_m2_winsor_mxn", "precio_m2_mxn_minmax", "precio_m2_mxn_zscore",
    "rango_superficie_cat", "estrato_precio_m2_cat", "nivel_vacancia_cat"
]].head(6)

=== Resumen de Normalización, Discretización y Jerarquías ===
{
  "variables_normalizadas_minmax_zscore": [
    "precio_m2_winsor_mxn",
    "superficie_m2",
    "renta_efectiva_winsor_mxn",
    "yield_bruto_anual_pct",
    "dias_vacancia_observados"
  ],
  "distribucion_rango_superficie": {
    "2_Medio (60-100 m2)": 425,
    "1_Compacto (<60 m2)": 361,
    "4_Residencial Plus (>160 m2)": 356,
    "3_Amplio (100-160 m2)": 149
  },
  "distribucion_estrato_precio_m2": {
    "Q4_Prime": 323,
    "Q2_Moderado": 323,
    "Q1_Accesible": 323,
    "Q3_Alto": 322
  },
  "distribucion_nivel_vacancia": {
    "Normal (22-35 dias)": 831,
    "Lenta (36-50 dias)": 261,
    "Rapida (<=21 dias)": 199
  },
  "niveles_jerarquia_espacial": [
    "colonia",
    "alcaldia_municipio",
    "subzona_nombre",
    "corredor_macro_zmvm",
    "region_metropolitana"
  ]
}


,id_propiedad,jerarquia_ubicacion_completa,trimestre,familia_inmueble,precio_m2_winsor_mxn,precio_m2_mxn_minmax,precio_m2_mxn_zscore,rango_superficie_cat,estrato_precio_m2_cat,nivel_vacancia_cat
0,PROP-ACC81F8D42,ZMVM (Valle de México) > Corredor Poniente Pri...,2023-Q4,Vivienda_Vertical_Departamento,436.58,0.5650,1.0052,2_Medio (60-100 m2),Q4_Prime,Normal (22-35 dias)
1,PROP-4E2A0FC1CC,ZMVM (Valle de México) > Corredor Poniente Pri...,2023-Q4,Vivienda_Vertical_Departamento,466.72,0.6339,1.3454,2_Medio (60-100 m2),Q4_Prime,Normal (22-35 dias)
2,PROP-898B72BF4C,ZMVM (Valle de México) > Corredor Poniente Pri...,2023-Q4,Vivienda_Vertical_Departamento,441.65,0.5766,1.0624,1_Compacto (<60 m2),Q4_Prime,Rapida (<=21 dias)
3,PROP-A7FA0E3263,ZMVM (Valle de México) > Corredor Poniente Pri...,2023-Q4,Vivienda_Vertical_Departamento,406.18,0.4955,0.6621,2_Medio (60-100 m2),Q3_Alto,Normal (22-35 dias)
4,PROP-39E1C13A7F,ZMVM (Valle de México) > Corredor Poniente Pri...,2023-Q4,Vivienda_Vertical_Departamento,408.63,0.5011,0.6897,3_Amplio (100-160 m2),Q3_Alto,Lenta (36-50 dias)
5,PROP-499C016032,ZMVM (Valle de México) > Corredor Poniente Pri...,2023-Q4,Vivienda_Vertical_Departamento,626.96,1.0000,3.1540,2_Medio (60-100 m2),Q4_Prime,Lenta (36-50 dias)


## 6. Reducción de Datos y Construcción de la Capa Oro (`data/gold/inmobiliario/`)
Se realiza la reducción en dos niveles:
1. **Selección de Variables (Reducción de Dimensionalidad):** De las **61 columnas** intermedias generadas en la Capa Plata completa, se seleccionan las **31 variables** canónicas sin redundancia (**reducción del $49.18\%$** en columnas).
2. **Agregación Multidimensional hacia Capa Oro:** Se generan los 3 datasets analíticos agregados para responder directamente las 3 preguntas de la Triléctica Estadística (`E1`, `E2` y `E3`).

In [7]:
gold_datasets, reduction_report = build_gold_layer_datasets(df_silver_full)
print("=== Reporte de Reducción de Datos y Tablas de Capa Oro ===")
print(json.dumps(reduction_report, indent=2, ensure_ascii=False))

print("\n--- [Capa Oro E1] Serie Mensual/Trimestral para Pronóstico (Primeros 5 periodos) ---")
display(gold_datasets["oro_e1_pronostico_mensual_trimestral"].head(5))

print("\n--- [Capa Oro E2] Cubo Experimental What-If Descuentos 24m (Primeras 6 combinaciones) ---")
display(gold_datasets["oro_e2_simulacion_whatif_24m"].head(6))

print("\n--- [Capa Oro E3] Cubo Comparación Multivariada Yield % (12 celdas Tipología x Sub-zona) ---")
display(gold_datasets["oro_e3_comparacion_multivariada_yield"])

=== Reporte de Reducción de Datos y Tablas de Capa Oro ===


{
  "columnas_totales_capa_plata_completa": 61,
  "columnas_seleccionadas_vista_reducida": 31,
  "reduccion_dimensionalidad_pct": 49.18,
  "filas_plata_microdatos": 1291,
  "filas_oro_e1_series_tiempo": 36,
  "filas_oro_e2_experimento_whatif": 28,
  "filas_oro_e3_multivariado_yield": 12
}

--- [Capa Oro E1] Serie Mensual/Trimestral para Pronóstico (Primeros 5 periodos) ---


,periodo_mensual,trimestre,anio,n_propiedades_validadas,precio_m2_media_mxn,precio_m2_mediana_mxn,precio_m2_std_mxn,renta_efectiva_mediana_mxn,yield_bruto_medio_pct,dias_vacancia_medio,inpc_vivienda_zmvm,inflacion_anual_vivienda_pct,tiie_28d_pct,tipo_cambio_fix_mxn_usd
0,2023-11,2023-Q4,2023,36,323.181,323.730,93.534,29965.185,6.444,29.028,121.400,4.65,11.500,17.150
1,2023-12,2023-Q4,2023,36,320.653,315.190,77.985,26824.575,6.499,31.361,121.932,4.75,11.466,17.191
2,2024-01,2024-Q1,2024,36,325.778,323.615,83.611,28269.910,6.417,30.944,122.608,4.84,11.427,17.139
3,2024-02,2024-Q1,2024,36,326.438,326.005,84.469,29315.540,6.356,27.417,123.278,4.93,11.380,16.987
4,2024-03,2024-Q1,2024,36,326.831,310.935,82.798,27272.845,6.248,28.833,123.792,5.00,11.320,16.819



--- [Capa Oro E2] Cubo Experimental What-If Descuentos 24m (Primeras 6 combinaciones) ---


,modalidad_contrato,descuento_contrato_pct,subzona_id,subzona_nombre,n_observaciones,dias_vacancia_promedio,dias_vacancia_mediana,tasa_ocupacion_24m_promedio_pct,renta_efectiva_promedio_mxn,yield_efectivo_promedio_pct,ingreso_efectivo_esperado_24m_mxn
0,largo_plazo_24m,0.0,SZ1_PRIME_PONIENTE,Prime Poniente (Miguel Hidalgo),20,38.35,37.0,94.75,45706.01,5.83,1039354.67
1,largo_plazo_24m,0.0,SZ2_CENTRAL_CORREDOR,Central Bohemio-Corporativa (Cuauhtémoc),22,29.27,28.5,95.99,41174.10,6.42,948552.45
2,largo_plazo_24m,0.0,SZ3_CONSOLIDADA_SUR,Consolidada Media-Alta (Benito Juárez / Coyoacán),31,32.26,33.0,95.58,31324.56,6.79,718560.35
3,largo_plazo_24m,0.0,SZ4_EMERGENTE_METROPOLITANA,En Consolidación / Conurbada (Sta. María - Esc...,24,35.92,35.0,95.08,25825.38,7.44,589314.51
4,largo_plazo_24m,5.0,SZ1_PRIME_PONIENTE,Prime Poniente (Miguel Hidalgo),22,28.41,27.5,96.11,41647.14,5.42,960649.59
5,largo_plazo_24m,5.0,SZ2_CENTRAL_CORREDOR,Central Bohemio-Corporativa (Cuauhtémoc),21,22.67,23.0,96.90,40813.80,6.17,949165.73



--- [Capa Oro E3] Cubo Comparación Multivariada Yield % (12 celdas Tipología x Sub-zona) ---


,subzona_id,subzona_nombre,familia_inmueble,tipo_inmueble,n_inmuebles,superficie_m2_promedio,precio_m2_promedio_mxn,precio_m2_mediana_mxn,renta_mensual_promedio_mxn,yield_bruto_promedio_pct,yield_bruto_mediana_pct,yield_bruto_std_pct,dias_vacancia_promedio
0,SZ1_PRIME_PONIENTE,Prime Poniente (Miguel Hidalgo),Vivienda_Horizontal_Casa,casa,108,200.55,377.70,376.85,73478.69,4.83,4.87,0.47,33.90
1,SZ1_PRIME_PONIENTE,Prime Poniente (Miguel Hidalgo),Vivienda_Vertical_Departamento,depto_1rec,108,52.29,491.58,490.94,24765.38,5.97,5.99,0.43,32.46
2,SZ1_PRIME_PONIENTE,Prime Poniente (Miguel Hidalgo),Vivienda_Vertical_Departamento,depto_2rec,107,87.36,444.49,436.03,37687.37,5.54,5.55,0.47,33.19
3,SZ2_CENTRAL_CORREDOR,Central Bohemio-Corporativa (Cuauhtémoc),Vivienda_Horizontal_Casa,casa,108,201.03,349.73,352.32,68118.94,5.60,5.62,0.45,25.50
4,SZ2_CENTRAL_CORREDOR,Central Bohemio-Corporativa (Cuauhtémoc),Vivienda_Vertical_Departamento,depto_1rec,107,52.62,455.54,457.78,23321.71,6.81,6.84,0.49,26.03
5,SZ2_CENTRAL_CORREDOR,Central Bohemio-Corporativa (Cuauhtémoc),Vivienda_Vertical_Departamento,depto_2rec,107,89.27,403.40,404.33,34854.21,6.31,6.36,0.44,25.36
6,SZ3_CONSOLIDADA_SUR,Consolidada Media-Alta (Benito Juárez / Coyoacán),Vivienda_Horizontal_Casa,casa,107,201.14,260.28,259.42,50831.75,5.99,6.04,0.43,27.47
7,SZ3_CONSOLIDADA_SUR,Consolidada Media-Alta (Benito Juárez / Coyoacán),Vivienda_Vertical_Departamento,depto_1rec,108,52.34,339.63,340.40,17331.24,7.23,7.26,0.46,28.23
8,SZ3_CONSOLIDADA_SUR,Consolidada Media-Alta (Benito Juárez / Coyoacán),Vivienda_Vertical_Departamento,depto_2rec,108,88.09,303.37,302.69,25797.39,6.68,6.70,0.47,26.85
9,SZ4_EMERGENTE_METROPOLITANA,En Consolidación / Conurbada (Sta. María - Esc...,Vivienda_Horizontal_Casa,casa,108,196.60,216.13,215.88,41550.48,6.66,6.70,0.50,32.02
